# 10 — Pipeline Observability Dashboard

Reads directly from `tlc_audit` (MongoDB) to provide a real-time view of pipeline health.

**Metrics covered:**
1. **Quarantine Rate** — `quarantined / (silver + quarantined)` per pipeline run
2. **Volumetric Traceability** — `Bronze IN == Silver OUT + Quarantine` (zero data loss proof)
3. **Processing Throughput** — `records_processed / duration_seconds` per run
4. **Quality Check Status** — Pass / Warning / Fail breakdown across all runs
5. **Error Timeline** — Any pipeline errors logged in the audit DB

> *"If the professor asks 'Where did this trip count come from?', we query `silver_run_id`
> in fact_trips → find the Silver run → its `bronze_run_id` → the exact Bronze batch → the
> raw Parquet file. Full chain in one query."*

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from core.storage.mongo_client import get_audit_db
from core.config.settings import settings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from pymongo import MongoClient
import warnings
warnings.filterwarnings("ignore")

# Styling
plt.rcParams.update({
    "figure.facecolor": "#0f172a",
    "axes.facecolor":   "#1e293b",
    "axes.edgecolor":   "#334155",
    "axes.labelcolor":  "#e2e8f0",
    "xtick.color":      "#94a3b8",
    "ytick.color":      "#94a3b8",
    "text.color":       "#e2e8f0",
    "grid.color":       "#334155",
    "grid.alpha":       0.5,
    "font.family":      "DejaVu Sans",
})

PALETTE = {
    "blue":   "#3b82f6",
    "green":  "#10b981",
    "orange": "#f59e0b",
    "red":    "#ef4444",
    "purple": "#8b5cf6",
    "teal":   "#06b6d4",
}

print("Imports OK. Connecting to tlc_audit...")

In [ ]:
# Connect to MongoDB audit database
client  = MongoClient(settings.mongo_uri(), serverSelectionTimeoutMS=5000)
audit_db = client[settings.MONGO_DB_AUDIT]

# ── Load pipeline_runs ────────────────────────────────────────────────────────
runs_cursor = audit_db["pipeline_runs"].find(
    {},
    {
        "execution_id":    1,
        "pipeline_name":   1,
        "status":          1,
        "start_time":      1,
        "end_time":        1,
        "duration_seconds":1,
        "output_summary":  1,
        "quality_checks":  1,
        "errors":          1,
        "_id":             0,
    }
)
runs_raw = list(runs_cursor)

# ── Load quarantine ────────────────────────────────────────────────────────────
quar_cursor = audit_db["quarantine"].find(
    {},
    {"pipeline": 1, "reason": 1, "silver_run_id": 1, "quarantined_at": 1, "_id": 0}
)
quar_raw = list(quar_cursor)

print(f"Pipeline runs loaded : {len(runs_raw)}")
print(f"Quarantine records   : {len(quar_raw):,}")

In [ ]:
# ── Build runs DataFrame ──────────────────────────────────────────────────────
runs = []
for r in runs_raw:
    summary = r.get("output_summary", {})
    qc_list = r.get("quality_checks", [])
    runs.append({
        "execution_id":    r.get("execution_id"),
        "pipeline":        r.get("pipeline_name"),
        "status":          r.get("status"),
        "start_time":      pd.to_datetime(r.get("start_time")),
        "duration_s":      r.get("duration_seconds", 0) or 0,
        "records_bronze":  summary.get("total_records_written", 0) or
                           summary.get("records_read_from_bronze", 0) or 0,
        "records_silver":  summary.get("records_written_to_silver", 0) or 0,
        "records_quar":    summary.get("records_quarantined", 0) or 0,
        "quar_rate_pct":   summary.get("quarantine_rate_pct", 0) or 0,
        "qc_passed":       sum(1 for qc in qc_list if qc.get("status") == "PASSED"),
        "qc_warning":      sum(1 for qc in qc_list if qc.get("status") == "WARNING"),
        "qc_failed":       sum(1 for qc in qc_list if qc.get("status") == "FAILED"),
        "error_count":     len(r.get("errors", [])),
    })

df_runs = pd.DataFrame(runs)
if not df_runs.empty:
    df_runs = df_runs.sort_values("start_time")
    # Throughput: records per second
    df_runs["throughput"] = np.where(
        df_runs["duration_s"] > 0,
        (df_runs["records_silver"] + df_runs["records_quar"]) / df_runs["duration_s"],
        0
    )

# ── Build quarantine DataFrame ────────────────────────────────────────────────
df_quar = pd.DataFrame(quar_raw) if quar_raw else pd.DataFrame(
    columns=["pipeline", "reason", "silver_run_id", "quarantined_at"]
)

print(f"Runs DataFrame: {len(df_runs)} rows")
if not df_runs.empty:
    display(df_runs[["execution_id", "pipeline", "status", "records_bronze",
                     "records_silver", "records_quar", "quar_rate_pct",
                     "duration_s", "throughput"]].to_string(index=False))

---
##  Dashboard — Pipeline Observability

In [ ]:
if df_runs.empty:
    print("No audit data found. Run the pipeline first (notebooks 01-05).")
else:
    fig = plt.figure(figsize=(20, 14), facecolor="#0f172a")
    fig.suptitle(
        "  NYC TLC Pipeline — Observability Dashboard",
        fontsize=18, fontweight="bold", color="#f8fafc", y=0.98
    )

    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # ── Panel 1: Quarantine Rate per Pipeline Run ──────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    silver_runs = df_runs[df_runs["pipeline"].str.startswith("silver")]
    if not silver_runs.empty:
        bars = ax1.bar(
            silver_runs["pipeline"],
            silver_runs["quar_rate_pct"],
            color=PALETTE["orange"], alpha=0.85, edgecolor="#f59e0b"
        )
        ax1.axhline(y=5, color=PALETTE["red"], linestyle="--", alpha=0.8, label="5% threshold")
        for bar, val in zip(bars, silver_runs["quar_rate_pct"]):
            ax1.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.05,
                f"{val:.2f}%", ha="center", va="bottom",
                fontsize=9, color="#f8fafc"
            )
        ax1.set_title("Quarantine Rate (%) per Silver Pipeline", fontsize=11, fontweight="bold")
        ax1.set_ylabel("Quarantine Rate (%)")
        ax1.legend()
        ax1.grid(axis="y")

    # ── Panel 2: Volumetric Traceability (Bronze == Silver + Quarantine) ────────
    ax2 = fig.add_subplot(gs[1, :2])
    silver_runs_v = df_runs[df_runs["records_bronze"] > 0]
    if not silver_runs_v.empty:
        x = np.arange(len(silver_runs_v))
        w = 0.28
        ax2.bar(x - w, silver_runs_v["records_bronze"],  w, label="Bronze IN",    color=PALETTE["teal"],   alpha=0.85)
        ax2.bar(x,     silver_runs_v["records_silver"],  w, label="Silver OUT",   color=PALETTE["blue"],   alpha=0.85)
        ax2.bar(x + w, silver_runs_v["records_quar"],    w, label="Quarantine",   color=PALETTE["orange"], alpha=0.85)
        ax2.set_xticks(x)
        ax2.set_xticklabels(silver_runs_v["pipeline"], rotation=20, ha="right")
        ax2.set_title("Volumetric Traceability\nBronze IN = Silver OUT + Quarantine", fontsize=11, fontweight="bold")
        ax2.set_ylabel("Record Count")
        ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
        ax2.legend(loc="upper right", fontsize=8)
        ax2.grid(axis="y")

        # Delta annotation (should always be 0)
        for i, row in enumerate(silver_runs_v.itertuples()):
            delta = int(row.records_bronze) - int(row.records_silver) - int(row.records_quar)
            color = PALETTE["green"] if delta == 0 else PALETTE["red"]
            ax2.text(i, max(row.records_bronze, row.records_silver, row.records_quar) * 1.02,
                     f"Δ={delta}", ha="center", color=color, fontsize=8, fontweight="bold")

    # ── Panel 3: Quality Check Status Donut ────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 2])
    qc_passed  = df_runs["qc_passed"].sum()
    qc_warning = df_runs["qc_warning"].sum()
    qc_failed  = df_runs["qc_failed"].sum()
    qc_total   = qc_passed + qc_warning + qc_failed

    if qc_total > 0:
        wedges, texts, autotexts = ax3.pie(
            [qc_passed, qc_warning, qc_failed],
            labels=[f"Passed\n({qc_passed})", f"Warning\n({qc_warning})", f"Failed\n({qc_failed})"],
            colors=[PALETTE["green"], PALETTE["orange"], PALETTE["red"]],
            autopct="%1.0f%%",
            startangle=90,
            wedgeprops=dict(width=0.5),  # donut
        )
        for at in autotexts:
            at.set_color("white")
            at.set_fontsize(9)
        ax3.set_title(f"Quality Check Status\n({qc_total} total checks)", fontsize=11, fontweight="bold")

    # ── Panel 4: Processing Throughput (records/s) ─────────────────────────────
    ax4 = fig.add_subplot(gs[2, :2])
    throughput_runs = df_runs[df_runs["throughput"] > 0]
    if not throughput_runs.empty:
        ax4.barh(
            throughput_runs["pipeline"],
            throughput_runs["throughput"],
            color=PALETTE["purple"], alpha=0.85
        )
        for i, (_, row) in enumerate(throughput_runs.iterrows()):
            ax4.text(
                row["throughput"] + 0.5, i,
                f"{row['throughput']:,.0f} rec/s  ({row['duration_s']:.1f}s)",
                va="center", fontsize=8, color="#e2e8f0"
            )
        ax4.set_title("Processing Throughput (records/second)", fontsize=11, fontweight="bold")
        ax4.set_xlabel("records / second")
        ax4.grid(axis="x")

    # ── Panel 5: Quarantine Reasons ────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 2])
    if not df_quar.empty and "reason" in df_quar.columns:
        reason_counts = df_quar["reason"].value_counts().head(6)
        ax5.barh(
            reason_counts.index,
            reason_counts.values,
            color=PALETTE["red"], alpha=0.75
        )
        ax5.set_title("Top Quarantine Reasons", fontsize=11, fontweight="bold")
        ax5.set_xlabel("Count")
        ax5.grid(axis="x")
    else:
        ax5.text(0.5, 0.5, "No quarantine data\navailable",
                 ha="center", va="center", fontsize=11,
                 color="#94a3b8", transform=ax5.transAxes)
        ax5.set_title("Top Quarantine Reasons", fontsize=11, fontweight="bold")

    plt.savefig("/home/jovyan/work/data/observability_dashboard.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Dashboard saved to /home/jovyan/work/data/observability_dashboard.png")

##  Lineage Trace Demo

Given a `silver_run_id` from `fact_trips`, reconstruct the full Bronze → Silver → Gold chain.

In [ ]:
# ── Pick the most recent Silver Yellow run ────────────────────────────────────
silver_run = audit_db["pipeline_runs"].find_one(
    {"pipeline_name": "silver_yellow", "status": "SUCCESS"},
    sort=[("start_time", -1)]
)

if silver_run:
    s_id = silver_run["execution_id"]
    print(f"══════════════════════════════════════════════════")
    print(f"  Lineage Trace for silver_run_id = '{s_id}'")
    print(f"══════════════════════════════════════════════════")
    print(f"  Pipeline     : {silver_run['pipeline_name']}")
    print(f"  Status       : {silver_run['status']}")
    print(f"  Started      : {silver_run['start_time']}")
    print(f"  Duration     : {silver_run.get('duration_seconds', '?')}s")
    summary = silver_run.get("output_summary", {})
    print(f"  Bronze IN    : {summary.get('records_read_from_bronze', '?'):,}")
    print(f"  Silver OUT   : {summary.get('records_written_to_silver', '?'):,}")
    print(f"  Quarantine   : {summary.get('records_quarantined', '?'):,}")
    print(f"  Quar Rate    : {summary.get('quarantine_rate_pct', '?'):.2f}%")

    # Find corresponding Bronze run from quarantine docs
    quar_sample = audit_db["quarantine"].find_one({"silver_run_id": s_id})
    if quar_sample:
        b_id = quar_sample.get("bronze_run_id")
        src_file = quar_sample.get("source_file")
        print(f"\n  ↓ Bronze lineage (from quarantine sample):")
        print(f"    bronze_run_id : {b_id}")
        print(f"    source_file   : {src_file}")
        print(f"\n  Full chain: {src_file} → [Bronze:{b_id}] → [Silver:{s_id}] → Gold")
    print(f"══════════════════════════════════════════════════")
else:
    print("No successful silver_yellow runs found. Run notebook 02 first.")

##  Recent Pipeline Runs Table

In [ ]:
import json

if not df_runs.empty:
    display_cols = [
        "execution_id", "pipeline", "status", "start_time",
        "duration_s", "records_bronze", "records_silver",
        "records_quar", "quar_rate_pct", "throughput", "error_count"
    ]
    print(df_runs[display_cols].to_string(index=False))
else:
    print("No pipeline runs in audit DB yet.")